In [45]:
!pip uninstall -y transformers
!pip install transformers==4.52.4

Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 124.7 MB/s eta 0:00:00


In [1]:
!pip uninstall -y torch
!pip install torch==2.6.0

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 108.4 MB/s eta 0:00:0

### 데이터 전처리

In [3]:
from google.colab import files
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
import torch
from datasets import Dataset

# 파일 업로드
uploaded = files.upload()

Saving cleaned_labeling_[SEP]Delete.csv to cleaned_labeling_[SEP]Delete (1).csv


In [4]:
# 데이터프레임으로 파일 읽기
df = pd.read_csv('/content/cleaned_labeling_[SEP]Delete.csv')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29521 entries, 0 to 29520
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    29521 non-null  object
 1   label   29521 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 461.4+ KB


In [6]:
df.head()

,text,label
0,0: Hola. 1: hi. 1: whats up? 0: not a ton. 0: ...,0
1,0: happy is ayuppie word. 0: a yuppie * 1: yup...,0
2,0: hi 1: hihi 1: m or f?? 0: akjfkjasdkjfajkdf...,0
3,0: Im a 17 F Lesbian. Im looking for girls onl...,0
4,0: hey! 1: Heyyyy 0: whats up 1: What&apos;s u...,0


In [7]:
import re

def parse_dialogue(dialogue_text):
    # speaker 구분이 '0:' 또는 '1:' 으로 되어있으므로 split
    # 단, 첫번째 '0:' 또는 '1:' 앞에 불필요한 공백 가능성 있음 → re 사용

    # '0:' 또는 '1:' 앞에서 split (keep separator)
    splits = re.split(r'(?=(?:0:|1:))', dialogue_text)

    # 빈 split 제거 + strip 적용
    turns = [s.strip() for s in splits if s.strip() != '']

    return turns

df['text'] = df['text'].apply(parse_dialogue)

In [8]:
df.head()

,text,label
0,"[0: Hola., 1: hi., 1: whats up?, 0: not a ton....",0
1,"[0: happy is ayuppie word., 0: a yuppie *, 1: ...",0
2,"[0: hi, 1: hihi, 1: m or f??, 0: akjfkjasdkjfa...",0
3,[0: Im a 17 F Lesbian. Im looking for girls on...,0
4,"[0: hey!, 1: Heyyyy, 0: whats up, 1: What&apos...",0


In [9]:
cnt = 0
a = []
for i in range(len(df['text'])):
  for turn in df['text'][i]:
    if turn.split(':')[0] != '0' and turn.split(':')[0] != '1':
      print(i, turn)
      a.append(i)
      cnt += 1
for i in a:
  print(i)
print(cnt)

223 ith a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re now chatting with a random stranger. Say hi!You&apos;re no

In [10]:
df.drop(a, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29509 entries, 0 to 29520
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    29509 non-null  object
 1   label   29509 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 691.6+ KB


In [11]:
df.reset_index(drop=True, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29509 entries, 0 to 29508
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    29509 non-null  object
 1   label   29509 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 461.2+ KB


In [12]:
cnt = 0
b = []
for i in range(len(df)):
  if df['label'][i] == 1:
    b.append(i)
    cnt += 1
print('총:', cnt)

tmp = []
for i in b:
  tmp.append(len(df['text'][i]))
print(sum(tmp) / len(tmp))
print(min(tmp))

총: 2570
77.64941634241245
10


In [12]:
t = pd.DataFrame(tmp)
t.head()

,0
0,97
1,311
2,47
3,20
4,25


In [18]:
t.value_counts()

,count
0,
10,35
23,33
28,33
11,32
21,29
...,...
745,1
746,1
779,1


In [42]:
tmp = []
for i in range(len(df)):
  if len(df['text'][i]) == 50 and df['label'][i] == 1:
    tmp.append(df['text'][i])

tmp[1]

['0: hey',
 '1: hello',
 '0: i liked talking to you for real',
 '1: I did too',
 '1: I masturbated foir you while talking to you',
 '0: who is sophia?',
 '1: My lawyer',
 '1: also my friend',
 '0: k',
 "0: i don't want to bug you",
 '1: you dont bug me',
 '0: no like you sound like really important',
 '1: what do you mean?',
 '0: on the phone your like really important',
 '1: hahahhahah',
 '0: what?',
 '1: you are funny',
 '0: i just dont want to bug you',
 '1: I enjoy you',
 '0: k I did something today',
 '1: tell me',
 '0: como estas',
 '1: great',
 '0: that means how are you',
 '1: perfect',
 '0: i tried',
 '1: nice',
 '0: ou did not say bein y tu',
 '1: that is suppose to be correct',
 '0: see I am smart',
 '1: That is why I like you',
 '0: ha ha i read it off the site...but will try',
 '1: it is OK. You have to start somehow',
 '0: i feel kinda freaked about panties',
 '1: mmmm',
 '1: dream about it',
 '0: no i dream to much and never comes true and what I dream bout girls panties

In [34]:
df['text'][0]

['0: Hola.',
 '1: hi.',
 '1: whats up?',
 '0: not a ton.',
 '0: you?',
 '1: same.  being lazy.  M or f?',
 '0: F.',
 '0: Ditto, I&apos;ve done absolutely nothing with my day besides watching stuff on Hulu.',
 '1: M here.  Just got home from weekend trip.  Tired.',
 '0: Oh, cool. Family thing?',
 '1: yeah.',
 '1: a &amp; l?',
 '0: Gotta love those.',
 '0: 17, Hawaii.',
 '0: and yourself?',
 '1: Uh oh.  older. 30',
 '1: Been to Hawaii.',
 '0: whoops xD',
 '0: It&apos;s nice, isn&apos;t it?',
 '1: Yeah.  Always enjoy visiting.',
 '1: Which Island you on Oahu?',
 '0: married?',
 '0: i&apos;m assuming since you went on a &apos;family&apos; trip :p',
 '1: yeah. Just found this site a few days ago.',
 '0: Yeah, Oahu.',
 '1: Curious to the whole &quot;random thing&quot;',
 '0: Pretty crazy the individuals you meet, isn&apos;t it?',
 '1: It&apos;s been eye opening for sure.',
 '0: Yeah, I hear you.',
 '1: I&apos;m pretty open to meet/talk to anyone.',
 '1: But pretty clear what most are looking

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29509 entries, 0 to 29508
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    29509 non-null  object
 1   label   29509 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 461.2+ KB


### 모델 구성

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import classification_report

# 학습용 데이터 구성

# window_size : 모델에 넣을 대화 흐름 길이 결정
def create_flow_inputs(turns_list, labels, window_size):
    input_texts = []
    target_labels = []

    for turns, label in zip(turns_list, labels):
        # 대화 turn 수가 window_size 미만이면 skip
        if len(turns) < window_size:
            continue

        # sliding window 적용 -> window_size 크기로 대화 잘라서 여러 개 샘플 생성
        # window가 10칸씩 앞으로 밀리면서 데이터 생성(0 ~ 20, 10 ~ 30, ...)
        for i in range(0, len(turns) - window_size + 1, 10):  # 10칸씩 이동
            window_turns = turns[i:i + window_size]           # window_size 만큼 자르기

            input_texts.append(window_turns)
            target_labels.append(label)

    return input_texts, target_labels

# 구성
input_texts, target_labels = create_flow_inputs(df['text'], df['label'], window_size=20)
input_texts = [' '.join(turns) for turns in input_texts]

# Huggingface Dataset 구성
from datasets import Dataset
dataset = Dataset.from_dict({'text': input_texts, 'label': target_labels})

# Tokenizer
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['text'], max_length=512, truncation=True, padding='max_length')

dataset = dataset.map(tokenize_function, batched=True)

# Train / Validation Split
train_test_split = dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split['train']
val_dataset = train_test_split['test']

# 6) Data collator (자동 padding)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 7) 모델 준비
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# 8) 학습 인자 설정
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

# 9) Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 10) 학습 시작
trainer.train()

# 11) 평가
predictions = trainer.predict(val_dataset)
labels = predictions.label_ids
preds = np.argmax(predictions.predictions, axis=-1)
print(classification_report(labels, preds))


Map:   0%|          | 0/81949 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-17-af76a7d0cc9c>:72: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,0.014700,0.019865


              precision    recall  f1-score   support

           0       1.00      0.99      1.00     13109
           1       0.98      0.99      0.99      3281

    accuracy                           0.99     16390
   macro avg       0.99      0.99      0.99     16390
weighted avg       0.99      0.99      0.99     16390



In [18]:
model.save_pretrained('/content/drive/MyDrive/디스부_최종프로젝트/Detection/BERT_SlidingWindow_2')
tokenizer.save_pretrained('/content/drive/MyDrive/디스부_최종프로젝트/Detection/BERT_SlidingWindow_2')

('/content/drive/MyDrive/디스부_최종프로젝트/Detection/BERT_SlidingWindow_2/tokenizer_config.json',
 '/content/drive/MyDrive/디스부_최종프로젝트/Detection/BERT_SlidingWindow_2/special_tokens_map.json',
 '/content/drive/MyDrive/디스부_최종프로젝트/Detection/BERT_SlidingWindow_2/vocab.txt',
 '/content/drive/MyDrive/디스부_최종프로젝트/Detection/BERT_SlidingWindow_2/added_tokens.json',
 '/content/drive/MyDrive/디스부_최종프로젝트/Detection/BERT_SlidingWindow_2/tokenizer.json')

### 평가

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

# 저장된 모델 경로 지정
model_path = '/content/drive/MyDrive/디스부_최종프로젝트/BERT_Sliding_Window'

# 모델 및 토크나이저 불러오기
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)

# GPU 사용 가능하면 이동
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
# 예측 테스트
#text = "0: hey u on? [SEP] 1: Hey baby [SEP] 1: What u doin [SEP] 0: hey [SEP] 1: Everything ok? [SEP] 0: um i guess so [SEP] 1: U have a good day? [SEP] 0: it was ok [SEP] 1: I missed u baby [SEP] 0: i broke a dish and my gramz is pissed off [SEP] 0: i miss u to [SEP] 1: Im sorry baby what dish was it? [SEP] 0: some china dish thing she had 4 like a 1000 years [SEP] 1: Oh [SEP] 1: That bites bad [SEP] 0: ya :( [SEP] 1: Wish i could make it better baby [SEP] 0: me too [SEP] 0: i hate this [SEP] 1: Wish u could just move in with me baby [SEP] 0: aww [SEP] 0: i wish that could happen [SEP] 0: it seems like 4eva until my dad gets back [SEP] 1: I can take care of u baby [SEP] 0: :(( [SEP] 1: Wish your dad could live down here near me so we can be together [SEP] 0: ya [SEP] 0: but he got a job in nj [SEP] 0: and i hate it here [SEP] 0: sorry [SEP] 1: I hope i would be a reason u wanna stay [SEP] 0: ya [SEP] 0: i cant wait til we meet [SEP] 1: I wish u would be my girl [SEP] 0: that wud b sweet [SEP] 1: Me either baby [SEP] 1: If your gramz leave for the day would it be better if u sneak off with me or we stay there at your place? [SEP] 0: idk [SEP] 0: maybe leave [SEP] 0: idk [SEP] 1: Ok [SEP] 0: if we stayed what u wanna do? [SEP] 1: I wish u were here now baby [SEP] 1: We could watch movies maybe make out or stuff [SEP] 0: if we left weher wud we go? [SEP] 1: Go shoppin for stuff for u then lunch then maybe go somewhere to be alone [SEP] 0: cool [SEP] 1: What u wanna shop for baby? [SEP] 0: what u wanna buy me?:-P [SEP] 1: How about some lingerie baby [SEP] 0: huh [SEP] 1: Lingerie is sexy bras and panties and thongs [SEP] 0: oh ok lol [SEP] 0: ya thatd b sweet [SEP] 1: I want my baby lookin very sexy [SEP] 0: o ya [SEP] 0: ima <3 that [SEP] 1: I want u to be if u want [SEP] 0: ya i do [SEP] 0: i need some fun [SEP] 1: U could have fun with me baby [SEP] 0: ya i want to [SEP] 1: :-* [SEP] 1: What all u wearin baby? [SEP] 0: jeans and a hoodie [SEP] 1: What kind bra and panties baby? [SEP] 0: lol um lemme look [SEP] 1: I could warm u up if i was with u [SEP] 0: blue stripe panties [SEP] 0: and a white bra [SEP] 0: my feet are always cold [SEP] 1: If u were here i would want u only in your bra and panties baby [SEP] 0: lol [SEP] 0: could i wear sox?:-P [SEP] 1: Yes but u may not need them baby [SEP] 0: awww [SEP] 0: u gonna rub my feet so there warm? [SEP] 1: We would be under a quilt makin out hot and heavy [SEP] 0: i think my feet would be cold neway [SEP] 0: lol [SEP] 1: Ill keep u warm all over baby [SEP] 0: i hope so [SEP] 1: Ill rub my legs over them warmin them up while we kiss [SEP] 0: ok <3 [SEP] 1: And my hands will rub u in other places warmin u up baby [SEP] 0: lol like where [SEP] 1: Your back, sides, your butt maybe in your panties [SEP] 0: oh [SEP] 1: You like baby? [SEP] 0: ya [SEP] 1: I want u baby [SEP] 0: u do? [SEP] 1: Yes do u want me baby? [SEP] 0: ya i do [SEP] 0: im lonely [SEP] 0: and i need sum1 to treat me nice [SEP] 1: Baby ill treat u like a princess [SEP] 0: ya [SEP] 0: i wanna b one again [SEP] 1: I hope u could tell that when u have talked to me [SEP] 0: ya [SEP] 1: Baby i would treat u as my princess when im with u baby [SEP] 0: u would be my prince? [SEP] 1: Yes baby [SEP] 1: Do u want me to be your prince? [SEP] 0: YA [SEP] 0: dang my gramz here [SEP] 0: i got to go now [SEP] 0: will u be on tomorow? [SEP] 1: Love ya baby [SEP] 0: :-* [SEP] 1: Yes baby [SEP] 1: :-*"
text = "0: hey u on? 1: Hey baby 1: What u doin 0: hey 1: Everything ok? 0: um i guess so 1: U have a good day? 0: it was ok 1: I missed u baby 0: i broke a dish and my gramz is pissed off 0: i miss u to 1: Im sorry baby what dish was it? 0: some china dish thing she had 4 like a 1000 years 1: Oh 1: That bites bad 0: ya :( 1: Wish i could make it better baby 0: me too 0: i hate this 1: Wish u could just move in with me baby 0: aww 0: i wish that could happen 0: it seems like 4eva until my dad gets back 1: I can take care of u baby 0: :(( 1: Wish your dad could live down here near me so we can be together 0: ya 0: but he got a job in nj 0: and i hate it here 0: sorry 1: I hope i would be a reason u wanna stay 0: ya 0: i cant wait til we meet 1: I wish u would be my girl 0: that wud b sweet 1: Me either baby 1: If your gramz leave for the day would it be better if u sneak off with me or we stay there at your place? 0: idk 0: maybe leave 0: idk 1: Ok 0: if we stayed what u wanna do? 1: I wish u were here now baby 1: We could watch movies maybe make out or stuff 0: if we left weher wud we go? 1: Go shoppin for stuff for u then lunch then maybe go somewhere to be alone 0: cool 1: What u wanna shop for baby? 0: what u wanna buy me?:-P 1: How about some lingerie baby  0: huh  1: Lingerie is sexy bras and panties and thongs  0: oh ok lol  0: ya thatd b sweet  1: I want my baby lookin very sexy  0: o ya  0: ima <3 that  1: I want u to be if u want  0: ya i do  0: i need some fun  1: U could have fun with me baby  0: ya i want to  1: :-*  1: What all u wearin baby?  0: jeans and a hoodie  1: What kind bra and panties baby?  0: lol um lemme look  1: I could warm u up if i was with u  0: blue stripe panties  0: and a white bra  0: my feet are always cold  1: If u were here i would want u only in your bra and panties baby  0: lol  0: could i wear sox?:-P  1: Yes but u may not need them baby  0: awww  0: u gonna rub my feet so there warm?  1: We would be under a quilt makin out hot and heavy  0: i think my feet would be cold neway  0: lol  1: Ill keep u warm all over baby  0: i hope so  1: Ill rub my legs over them warmin them up while we kiss  0: ok <3  1: And my hands will rub u in other places warmin u up baby  0: lol like where  1: Your back, sides, your butt maybe in your panties 0: oh 1: You like baby? 0: ya 1: I want u baby 0: u do? 1: Yes do u want me baby? 0: ya i do 0: im lonely 0: and i need sum1 to treat me nice 1: Baby ill treat u like a princess 0: ya 0: i wanna b one again 1: I hope u could tell that when u have talked to me 0: ya 1: Baby i would treat u as my princess when im with u baby 0: u would be my prince? 1: Yes baby 1: Do u want me to be your prince? 0: YA 0: dang my gramz here 0: i got to go now 0: will u be on tomorow? 1: Love ya baby 0: :-* 1: Yes baby 1: :-*"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

outputs = model(**inputs)
logits = outputs.logits
predicted_class = logits.argmax().item()

print(f"Predicted class: {predicted_class} (0=non-grooming, 1=grooming)")


Predicted class: 1 (0=non-grooming, 1=grooming)
